In [121]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.temporal_features import create_temporal_features
from src.weather_features import create_weather_features

In [97]:
df = pd.read_csv('../data/raw/Metro_Interstate_Traffic_Volume.csv')
print(f"Dataset chargé : {len(df):,} lignes × {len(df.columns)} colonnes")

Dataset chargé : 48,204 lignes × 9 colonnes


Gestion des valeurs manquantes

In [99]:
if 'holiday' in df.columns:
    missing_before = df['holiday'].isnull().sum()
    df['holiday'] = df['holiday'].fillna('None')
    print(f"'holiday' : {missing_before:,} NaN remplacés par 'None'")

remaining_missing = df.isnull().sum().sum()
if remaining_missing > 0:
    print(f"{remaining_missing} valeurs manquantes restantes")
    df = df.dropna(subset=['traffic_volume', 'temp', 'date_time'])
    print(f"Lignes avec NaN critiques supprimées")
else:
    print(f"Aucune valeur manquante restante")

'holiday' : 48,143 NaN remplacés par 'None'
Aucune valeur manquante restante


Dans un premier temps, nous avons remplacé toutes les valeurs manquantes (NaN) par la chaîne de caractères "None"
Ensuite, nous avons transformé cette variable catégorielle en variable binaire "is_holiday" 

Suppression des doublons

In [101]:
print("Doublons :")

duplicates_count = df.duplicated().sum()
if duplicates_count > 0:
    print(f"{duplicates_count} doublons détectés")
    df = df.drop_duplicates()
    print(f"{duplicates_count} doublons supprimés")
else:
    print(f"Aucun doublon détecté")

Doublons :
17 doublons détectés
17 doublons supprimés


Suppression des outliers

In [103]:
print("Outliers :")
# Outlier 1 : rain_1h > 400 mm
if 'rain_1h' in df.columns:
    rain_outliers = (df['rain_1h'] > 100).sum()
    if rain_outliers > 0:
        print(f"rain_1h > 400mm : {rain_outliers} valeurs aberrantes")
        df = df[df['rain_1h'] <= 400]
        print(f"{rain_outliers} outliers de pluie supprimés")
    else:
        print(f"Aucun outlier dans 'rain_1h'")
# Outlier 2 : temp = 0K
if 'temp' in df.columns:
    temp_outliers = (df['temp'] == 0).sum()
    if temp_outliers > 0:
        print(f"temp = 0K : {temp_outliers} valeurs impossibles")
        df = df[df['temp'] > 0]
        print(f"{temp_outliers} températures impossibles supprimées")
    else:
        print(f"Aucune température impossible")
df = df.reset_index(drop=True)

print(f"\nAprès nettoyage : {len(df):,} lignes × {len(df.columns)} colonnes")

Outliers :
rain_1h > 400mm : 1 valeurs aberrantes
1 outliers de pluie supprimés
temp = 0K : 10 valeurs impossibles
10 températures impossibles supprimées

Après nettoyage : 48,176 lignes × 9 colonnes


À partir de rain_1h = 400 mm/h, c'est physiquement IMPOSSIBLE (au-delà du record mondial absolu).
temp = 0 Kelvin (-273°C).Cette température est physiquement impossible sur Terre.

Le nettoyage final consiste donc à supprimer uniquement 11 lignes sur les 48 204 lignes totales (0.02%), ce qui préserve l'intégrité de nos données tout en éliminant les erreurs de mesure.


Conversion température de kelvin au celsius

In [105]:
if 'temp_celsius' not in df.columns:
    df['temp_celsius'] = df['temp'] - 273.15
    print(f"Colonne 'temp_celsius' créée")
    print(f"Exemple : {df['temp'].iloc[0]:.2f}K = {df['temp_celsius'].iloc[0]:.2f}°C")
else:
    print(f"Colonne 'temp_celsius' existe déjà")

print(f"\nStatistiques température :")
print(f"   • Min     : {df['temp_celsius'].min():.2f}°C")
print(f"   • Max     : {df['temp_celsius'].max():.2f}°C")
print(f"   • Moyenne : {df['temp_celsius'].mean():.2f}°C")

Colonne 'temp_celsius' créée
Exemple : 288.28K = 15.13°C

Statistiques température :
   • Min     : -29.76°C
   • Max     : 36.92°C
   • Moyenne : 8.11°C


Feature Engineering 

Le Feature Engineering consiste à créer de nouvelles variables à partir des données brutes pour améliorer la capacité prédictive des modèles. Dans notre mini  projet de prédiction du trafic routier, nous avons créé deux catégories de features : temporelles et météorologiques.

Pour faciliter la maintenance et la réutilisabilité du code, nous avons organisé nos fonctions de feature engineering dans deux modules Python séparés temporal_features.py et weather_features.py.

Feature Engineering Temporel

In [107]:
df = create_temporal_features(df, date_column='date_time')

temporal_features = ['year', 'month', 'day', 'hour', 'day_of_week', 
                     'week_of_year', 'is_weekend', 'is_rush_hour', 
                     'time_of_day', 'season', 'is_holiday']

print(f"Features temporelles ajoutées :")
for feat in temporal_features:
    if feat in df.columns:
        print(f"   • {feat}")

Extraction des composantes temporelles:
Features temporelles ajoutées :
   • year
   • month
   • day
   • hour
   • day_of_week
   • week_of_year
   • is_weekend
   • is_rush_hour
   • time_of_day
   • season
   • is_holiday


Feature Engineering Temporel
Nous avons créé 11 features temporelles à partir de date_time :
year, month, day, hour, day_of_week, week_of_year : Composantes basiques extraites directement de la date pour capturer les cycles temporels.
autre features :
is_weekend : 1 si samedi/dimanche, 0 sinon. Le trafic du week-end est très différent des jours ouvrables.
is_rush_hour : 1 si 7h, 8h, 15h, 16h, 17h, 18h, 0 sinon. Ces heures montrent des pics de trafic massifs (trajets domicile-travail).
time_of_day : Catégories "Nuit", "Matin", "Après-midi", "Soirée". Simplifie les 24 heures en périodes significatives.
season : "Hiver", "Printemps", "Été", "Automne". Capture l'impact des saisons sur le trafic.
is_holiday : 1 si jour férié, 0 sinon. Les jours fériés réduisent le trafic de 40-50%.

Feature Engineering Météorologique

In [109]:
df = create_weather_features(df)

weather_features = ['is_raining', 'is_snowing', 'temp_level', 'cloud_category']

print(f"Features météo ajoutées :")
for feat in weather_features:
    if feat in df.columns:
        print(f"   • {feat}")

Création de 'is_raining' et 'is_snowing' :
Création de 'temp_level':
Création de 'cloud_category':
Feature engineering météo terminé ! 25 colonnes au total.
Features météo ajoutées :
   • is_raining
   • is_snowing
   • temp_level
   • cloud_category


Nous avons créé 4 features météorologiques à partir des variables météo existantes.
is_raining : 1 si pluie, 0 sinon. Simplifie la détection de pluie.
is_snowing : 1 si neige, 0 sinon. Simplifie la détection de neige.
temp_level : Catégories de température ("Très froid", "Froid", "frais", "agréable", "chaud").
cloud_category : Catégories de couverture nuageuse ("Ciel Dégagé", "Peu nuageux", "Partiellement Nuageux","Nuageux","Très nuageux").

Encodage des variables catégorielles

In [111]:
exclude_cols = ['date_time', 'date']
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col not in exclude_cols]

print(f"\nColonnes catégorielles à encoder : {categorical_cols}")

if len(categorical_cols) > 0:
    df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    
    new_cols = [col for col in df_encoded.columns if col not in df.columns]
    print(f"\nOne-Hot Encoding appliqué")
    print(f"{len(new_cols)} nouvelles colonnes créées")
    print(f"Exemples : {new_cols[:5]}")
    
    df = df_encoded
else:
    print("Aucune colonne catégorielle à encoder")


Colonnes catégorielles à encoder : ['holiday', 'weather_main', 'weather_description', 'time_of_day', 'season', 'temp_level', 'cloud_category']

One-Hot Encoding appliqué
72 nouvelles colonnes créées
Exemples : ['holiday_Columbus Day', 'holiday_Independence Day', 'holiday_Labor Day', 'holiday_Martin Luther King Jr Day', 'holiday_Memorial Day']


Les algorithmes de machine learning ne peuvent pas traiter directement les variables textuelles. Nous avons donc appliqué le One-Hot Encoding pour convertir les 7 variables catégorielles en variables numériques binaires.

Sauvegarde de dataset final 

In [117]:
output_dir = os.path.abspath("../processed")
os.makedirs(output_dir, exist_ok=True)

filename = os.path.join(output_dir, "dataset_final_cleaned.csv")
df.to_csv(filename, index=False)

print(f"   -> {len(df):,} lignes")
print(f"   -> {len(df.columns)} colonnes")
print(f"   -> Taille : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


   -> 48,176 lignes
   -> 90 colonnes
   -> Taille : 7.95 MB


Apercu de dataset final 

In [ ]:
print("Premières lignes :")
print(df.head(5))

print("\nStatistiques de la target (traffic_volume) :")
print(f"   -> Moyenne   : {df['traffic_volume'].mean():.0f} véh/h")
print(f"   -> Médiane   : {df['traffic_volume'].median():.0f} véh/h")
print(f"   -> Min       : {df['traffic_volume'].min():.0f} véh/h")
print(f"   -> Max       : {df['traffic_volume'].max():.0f} véh/h")
print(f"   -> Écart-type: {df['traffic_volume'].std():.0f} véh/h")

Premières lignes :
     temp  rain_1h  snow_1h  clouds_all           date_time  traffic_volume  \
0  288.28      0.0      0.0          40 2012-10-02 09:00:00            5545   
1  289.36      0.0      0.0          75 2012-10-02 10:00:00            4516   
2  289.58      0.0      0.0          90 2012-10-02 11:00:00            4767   
3  290.13      0.0      0.0          90 2012-10-02 12:00:00            5026   
4  291.14      0.0      0.0          75 2012-10-02 13:00:00            4918   

   temp_celsius  year  month  day  ...  season_Printemps  season_Été  \
0         15.13  2012     10    2  ...             False       False   
1         16.21  2012     10    2  ...             False       False   
2         16.43  2012     10    2  ...             False       False   
3         16.98  2012     10    2  ...             False       False   
4         17.99  2012     10    2  ...             False       False   

   temp_level_Chaud  temp_level_Frais  temp_level_Froid  \
0             